<a href="https://colab.research.google.com/github/craljimenez/An-lisis_y_Visualizacion_de_Datos_para_la_Toma_de_Decisiones/blob/main/clases/semana04-clase08-transformacion-enriquecimiento-datos/notebooks/lab-transformacion-enriquecimiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio: Transformación y enriquecimiento de datos
### Semana 4 · Clase 8
**Curso:** Análisis y Visualización de Datos para la Toma de Decisiones (Esumer)

En la Clase 7 auditamos `membresias_gimnasio.csv` y encontramos cuatro tipos de problemas: **nulos**, **duplicados**, **tipos incorrectos** (`telefono` como número, `edad` con texto mezclado) e **inconsistencias de texto** (`ciudad` y `activo` con muchas formas de escribir lo mismo). Quedó pendiente corregirlos: ese es el plan de hoy.

Además de corregir, vamos a **enriquecer** la tabla: crear columnas nuevas que no estaban en el archivo original, pero que se pueden calcular a partir de lo que sí tenemos (una antigüedad en días, un grupo de edad, el precio que paga el socio según su plan). Esas columnas nuevas son las que te van a servir la próxima clase para agrupar y comparar.

## 0. Un principio antes de empezar: pipeline reproducible

Vas a transformar la tabla en varios pasos, uno detrás del otro. Dos reglas simples evitan un dolor de cabeza enorme más adelante:

1. **Nunca edites el archivo crudo.** Todo lo que hagas hoy parte de `pd.read_csv("membresias_gimnasio.csv")` sin tocar ese archivo; si te equivocas en un paso, puedes volver a leerlo desde cero y no perdiste nada.
2. **Evita cualquier cosa que dé un resultado distinto cada vez que la ejecutas.** Por ejemplo, calcular una antigüedad con "la fecha de hoy" (`pd.Timestamp.now()`) hace que tu resultado cambie según el día que lo corras; nadie más podría reproducir exactamente tus números. En su lugar, vamos a fijar una **fecha de referencia explícita**, igual que en la Clase 5 fijamos una semilla (`SEED`) para que la base de datos fuera siempre la misma.

Esa es la idea de un **pipeline reproducible**: una secuencia de pasos, siempre en el mismo orden, que cualquiera puede volver a correr y obtener exactamente el mismo resultado.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

socios = pd.read_csv("membresias_gimnasio.csv")
socios.shape

Fecha de referencia para todos los cálculos de "antigüedad" de hoy (la fijamos nosotros, no dejamos que Python la tome del reloj del computador):

In [ ]:
FECHA_REFERENCIA = pd.Timestamp("2026-09-30")

## 1. Paso 1: quitar los duplicados

De la Clase 7 ya sabes que hay filas exactamente repetidas. Es el primer paso porque, si no lo haces primero, cada estadística que calcules después (promedios, conteos) va a estar inflada por las copias.

In [ ]:
print("Filas antes:", len(socios))
socios = socios.drop_duplicates().reset_index(drop=True)
print("Filas después:", len(socios))

`reset_index(drop=True)` reacomoda el índice por defecto (0, 1, 2...) después de haber borrado filas, si no lo haces, el índice queda con "huecos" donde estaban las filas eliminadas.

## 2. Paso 2: corregir las inconsistencias de texto

Esto es lo que quedó pendiente de la Sección 6.4 de la Clase 7. La herramienta es encadenar **métodos vectorizados de texto** (todos empiezan con `.str`, se aplican a la columna completa de una sola vez, sin ciclos) y un **mapeo de valores** para los casos que ni con mayúsculas/minúsculas se arreglan.

In [ ]:
socios["ciudad"].str.strip().str.lower().value_counts()

`.str.strip()` quita espacios sobrantes al inicio/final; `.str.lower()` pasa todo a minúsculas. Ya bajamos de 12 variantes a solo 6, pero **`bogota` sigue separado de `bogotá`**, y **`medellin` de `medellín`**: quitar mayúsculas y espacios no agrega ni quita tildes. Para esos, hace falta decirle explícitamente a pandas que son el mismo valor, con `.replace()` y un diccionario:

In [ ]:
socios["ciudad_limpia"] = (
    socios["ciudad"]
    .str.strip()
    .str.lower()
    .replace({"bogota": "bogotá", "medellin": "medellín", "itagui": "itagüí"})
    .str.title()
)
socios["ciudad_limpia"].value_counts()

`.str.title()` al final pone en mayúscula la primera letra de cada palabra, para que quede con formato de nombre propio (`"medellín"` → `"Medellín"`). Ahora sí: 4 ciudades reales, ni una más.

Nota que guardamos el resultado en una columna **nueva** (`ciudad_limpia`), sin tocar `ciudad` todavía. Así puedes comparar las dos y confirmar que la limpieza hizo lo que esperabas, antes de reemplazar nada.

El mismo patrón, ahora para `activo` (que tenía 9 formas distintas de decir sí/no):

In [ ]:
socios["activo"].str.strip().str.lower().value_counts()

In [ ]:
socios["activo_limpio"] = (
    socios["activo"]
    .str.strip()
    .str.lower()
    .map({"si": True, "sí": True, "1": True, "no": False, "0": False})
)
socios["activo_limpio"].value_counts()

Aquí usamos `.map()` en vez de `.replace()` porque no solo estamos *corrigiendo texto*, estamos **cambiando de tipo de dato**: de texto a `bool` (`True`/`False`), un tipo mucho más útil para filtrar y contar más adelante (`socios[socios["activo_limpio"]]` te da directamente los socios activos, sin comparar contra ningún texto).

Actividad: revisa `socios["plan"].str.strip().str.lower().value_counts()`. En la Clase 7 viste que `plan` ya estaba limpio, confírmalo aquí también, para acostumbrarte a nunca asumir que una columna está bien sin mirarla.

In [ ]:
# TODO: confirma que "plan" no tiene inconsistencias


## 3. Paso 3: corregir los tipos incorrectos

### `telefono`: de número a texto

En la Clase 7 viste que `telefono` quedó como `float64` y se mostraba en notación científica. Un teléfono no es una cantidad: es texto que solo tiene dígitos. La forma correcta de arreglarlo tiene dos pasos: primero a un tipo entero que sí admite nulos (`Int64`, con "I" mayúscula, distinto del `int64` normal, que no puede tener `NaN`), y de ahí a texto.

In [ ]:
socios["telefono_limpio"] = socios["telefono"].astype("Int64").astype(str).replace("<NA>", pd.NA)
socios[["nombre", "telefono", "telefono_limpio"]].head(3)

`astype("Int64")` quita el `.0` sin pasar por texto primero (si intentas `.astype(str)` directo sobre el `float64`, el resultado conserva el `.0`: pruébalo tú mismo y compara). Los teléfonos que estaban vacíos se mantienen vacíos (`pd.NA`), no se inventan un valor.

### `edad`: de texto a número, con una decisión de negocio

`pd.to_numeric(..., errors="coerce")` (ya la usaste en la Clase 7) convierte lo que sí es número y vuelve `NaN` el resto, incluyendo los valores como `"treinta"`, que son texto disfrazado de número, no un dato vacío de verdad.

In [ ]:
edad_numerica = pd.to_numeric(socios["edad"], errors="coerce")
print("Valores sin convertir (vacíos + texto raro):", edad_numerica.isna().sum())

Aquí hay que tomar una decisión: ¿qué hacemos con esas filas? Las opciones más comunes son **eliminarlas** (si son pocas y no es grave perderlas) o **imputarlas** (rellenarlas con un valor razonable, típicamente la mediana, que no se deja arrastrar por valores extremos como el promedio). Para este laboratorio, imputamos con la mediana, pero la decisión correcta depende del caso, y siempre debe quedar documentada, no solo ejecutada:

In [ ]:
mediana_edad = edad_numerica.median()
socios["edad_limpia"] = edad_numerica.fillna(mediana_edad).astype(int)
print(f"Mediana usada para imputar: {mediana_edad}")
socios["edad_limpia"].describe()

Actividad: ¿cuántas filas de `socios` tienen `edad_limpia` igual a la mediana imputada? (pista: cuenta cuántas veces se repite ese valor con `.value_counts()` o compara `edad_numerica.isna()`). Es un buen chequeo para no olvidar cuántos valores fueron, en realidad, inventados por ti y no reportados por el socio.

In [ ]:
# TODO: cuenta cuántas filas quedaron con la edad imputada


## 4. Paso 4: fechas y periodos

`fecha_inscripcion` llegó como texto. Conviértela a fecha de verdad (como en la Clase 4) y usa el resultado para derivar información que **no estaba directamente en el archivo**: hace cuántos días se inscribió cada socio, y en qué trimestre.

In [ ]:
socios["fecha_inscripcion"] = pd.to_datetime(socios["fecha_inscripcion"])
socios["fecha_inscripcion"].dtype

In [ ]:
socios["antiguedad_dias"] = (FECHA_REFERENCIA - socios["fecha_inscripcion"]).dt.days
socios["trimestre_inscripcion"] = socios["fecha_inscripcion"].dt.quarter
socios[["nombre", "fecha_inscripcion", "antiguedad_dias", "trimestre_inscripcion"]].head(5)

`(FECHA_REFERENCIA - fecha)` resta dos fechas y da un `Timedelta` (una duración); `.dt.days` lo convierte en un número entero de días, exactamente el mismo tipo de resta de fechas que viste en el reto de la Clase 4, ahora aplicada a toda la columna de una vez. `.dt.quarter` es uno de varios atributos de fecha útiles (también existen `.dt.year`, `.dt.month`, `.dt.day_name()`, entre otros).

## 5. Paso 5: funciones sobre columnas, `.apply()`, `.map()` y operaciones vectorizadas

A veces la transformación que necesitas no tiene un método de pandas ya hecho. Ahí entran `.apply()` y `.map()`: les pasas una función (puede ser una que tú escribas) y pandas la ejecuta sobre cada valor de la columna.

In [ ]:
def primer_nombre(nombre_completo):
    return nombre_completo.split()[0]

socios["primer_nombre"] = socios["nombre"].apply(primer_nombre)
socios[["nombre", "primer_nombre"]].head(3)

Esto funciona, pero casi siempre que hay una operación de texto column-a-columna, pandas ya trae una versión **vectorizada** que hace lo mismo, y suele ser más rápida y más clara de leer, porque no necesitas definir una función aparte:

In [ ]:
socios["primer_nombre_v2"] = socios["nombre"].str.split(" ").str[0]

# confirmamos que dan exactamente el mismo resultado
(socios["primer_nombre"] == socios["primer_nombre_v2"]).all()

### `.map()`: el primo de `.apply()` para una sola columna

`Series.map()` hace lo mismo que `.apply()` sobre una columna: recibe una función y la ejecuta valor por valor.

In [ ]:
socios["primer_nombre_map"] = socios["nombre"].map(primer_nombre)

# .apply() y .map() con la misma función dan exactamente el mismo resultado
(socios["primer_nombre"] == socios["primer_nombre_map"]).all()

De hecho, ya usaste `.map()` antes, en la Sección 2 (`activo_limpio`) y lo vas a volver a usar en la Sección 6 (`precio_plan_cop`). Ahí no le pasaste una función, sino un **diccionario**. Esa es la otra mitad de lo que sabe hacer `.map()`: además de funciones, acepta un diccionario (o una `Series`) para **sustituir** cada valor por su equivalente, sin necesidad de escribir ninguna función.

En la práctica, para aplicar una función a una sola columna, `.map()` y `.apply()` son intercambiables: verás las dos formas en código de otras personas. La diferencia real aparece cuando la lógica que necesitas ya no cabe en una sola columna.

### Cuando una columna no alcanza: `.apply()` sobre filas completas

¿Qué pasa si la regla que necesitas depende de **varias columnas a la vez**? Por ejemplo: *"marca para seguimiento a los socios inactivos que se inscribieron hace menos de 120 días"*. Esa regla necesita `activo_limpio` **y** `antiguedad_dias` al mismo tiempo. Ni `.map()` ni el `.apply()` de una `Series` pueden hacer esto, porque los dos trabajan sobre una sola columna. Para esto existe `DataFrame.apply(función, axis=1)`: en vez de pasarle un valor a la vez, le pasa **una fila completa** a la función.

In [ ]:
def requiere_seguimiento(fila):
    return (not fila["activo_limpio"]) and fila["antiguedad_dias"] < 120

socios["requiere_seguimiento"] = socios.apply(requiere_seguimiento, axis=1)
socios["requiere_seguimiento"].value_counts()

In [ ]:
socios.loc[socios["requiere_seguimiento"], ["nombre", "activo_limpio", "antiguedad_dias"]]

`axis=1` le dice a `.apply()` que recorra **filas** (con `axis=0`, el valor por defecto, recorrería columnas completas en vez de filas; casi nunca es lo que quieres para este tipo de regla). Dentro de la función, `fila["activo_limpio"]` y `fila["antiguedad_dias"]` acceden a los valores de esa fila por nombre de columna, como si `fila` fuera un mini-diccionario.

| Herramienta | Opera sobre | Recibe | Úsala cuando... |
|---|---|---|---|
| Vectorizado (`.str`, `.dt`, `+`, `*`) | una columna completa | (ninguno) | exista esa operación ya hecha (primera opción, siempre) |
| `.map()` | una columna, valor por valor | función, o diccionario/Series | necesitas sustituir o transformar **una sola columna** |
| `.apply()` sobre una `Series` | una columna, valor por valor | función | igual que `.map()` con función: elige el que te resulte más claro |
| `.apply(..., axis=1)` sobre un `DataFrame` | filas completas | función que recibe la fila | la regla necesita **combinar varias columnas a la vez** |

In [ ]:
socios = socios.drop(columns=["primer_nombre_map"])  # ya cumplió su propósito de comparación

**Regla práctica:** si existe una operación vectorizada (`.str...`, `.dt...`, operadores aritméticos directos como `+`, `*`) que resuelva lo que necesitas, úsala primero: es la forma "nativa" de pandas de operar sobre una columna completa sin ciclos, la misma idea que ya viste en la Clase 4 con `df["total"] = df["cantidad"] * df["precio_unitario"]`. Reserva `.apply()` para cuando la lógica es demasiado particular como para expresarla con los métodos ya existentes.

In [ ]:
socios = socios.drop(columns=["primer_nombre_v2"])  # ya cumplió su propósito de comparación

## 6. Paso 6: enriquecer con conocimiento de negocio

Hasta ahora solo hemos *corregido* columnas que ya existían. Enriquecer es distinto: es agregar información nueva que no viene en el archivo, pero que conoces por otra vía, en este caso, el precio de cada plan (un dato que vive en la lista de precios del gimnasio, no en este CSV).

In [ ]:
precio_por_plan = {
    "Mensual": 90000,
    "Trimestral": 240000,
    "Anual": 800000,
}
socios["precio_plan_cop"] = socios["plan"].map(precio_por_plan)
socios[["nombre", "plan", "precio_plan_cop"]].head(5)

`.map()` busca, para cada valor de `plan`, la llave correspondiente en el diccionario y devuelve su valor: el mismo mecanismo que usaste para `activo_limpio`, ahora para añadir un dato nuevo en vez de corregir uno existente. Si algún plan no estuviera en el diccionario, esa fila quedaría en `NaN`: una forma automática de detectar valores inesperados (más sobre integrar tablas completas en la Clase 10, con `merge`).

Ojo con el significado de esta columna: cada precio es lo que se paga **por el plan completo** (90.000 por un mes, 240.000 por tres meses, 800.000 por un año), no un precio mensual. Sumar o comparar esos valores tal cual mezcla periodos distintos; llevarlos a una misma escala es justo lo que haremos en la Clase 9.

## 7. Paso 7: categorización: crear un grupo de edad

Una variable numérica como `edad_limpia` a veces es más útil para el análisis si la conviertes en **categorías** con significado de negocio. `pd.cut()` corta un rango numérico en "cajones" (*bins*) que tú defines.

In [ ]:
socios["grupo_edad"] = pd.cut(
    socios["edad_limpia"],
    bins=[0, 25, 45, 120],
    labels=["Joven (<=25)", "Adulto (26-45)", "Mayor (46+)"],
)
socios["grupo_edad"].value_counts()

`bins=[0, 25, 45, 120]` define tres rangos (de 0 a 25, de 25 a 45, y de 45 a 120) y `labels` les pone nombre a cada uno. Esta es exactamente el tipo de **variable analítica para segmentar** socios que vas a usar la próxima clase con `groupby`, para comparar (por ejemplo) cuánto gasta en promedio cada grupo de edad.

Actividad: crea una categorización parecida, pero para `antiguedad_dias`: dos grupos, `"Nuevo (< 180 días)"` y `"Antiguo (>= 180 días)"`. Guárdala en una columna `antiguedad_categoria`.

In [ ]:
# TODO: pd.cut() sobre antiguedad_dias con dos categorías


## 8. Paso 8: renombrar variables y armar la tabla final

Ya tienes las versiones corregidas de `ciudad`, `activo`, `telefono` y `edad` guardadas aparte (con sufijo `_limpia`/`_limpio`), más las columnas nuevas de enriquecimiento. Para cerrar el pipeline: sacamos las versiones crudas y le devolvemos a las columnas limpias su nombre original, con `.drop()` y `.rename()`.

In [ ]:
socios_limpio = (
    socios
    .drop(columns=["ciudad", "telefono", "edad", "activo", "primer_nombre"])
    .rename(columns={
        "ciudad_limpia": "ciudad",
        "telefono_limpio": "telefono",
        "edad_limpia": "edad",
        "activo_limpio": "activo",
    })
)
socios_limpio.head()

In [ ]:
socios_limpio.dtypes

Compara esto con el `socios.dtypes` de la Clase 7: `ciudad` y `activo` cambiaron de tipo (`activo` ahora es `bool` de verdad), `telefono` es texto limpio, `edad` es un número entero sin valores raros, `fecha_inscripcion` es una fecha real, y aparecieron seis columnas que no existían en el archivo original (`antiguedad_dias`, `trimestre_inscripcion`, `precio_plan_cop`, `grupo_edad`, `requiere_seguimiento`, y la que tú creaste en la actividad anterior).

### Guardar el resultado

Por el mismo principio de la Sección 0: **nunca sobrescribas el archivo crudo**. Guarda el resultado limpio con otro nombre.

In [ ]:
socios_limpio.to_csv("membresias_gimnasio_limpio.csv", index=False)
print("Guardado:", socios_limpio.shape)

Descarga el archivo a tu computador con el panel de archivos de Colab (ícono de carpeta, a la izquierda), por si quieres conservarlo. Recuerda que, igual que todo lo demás en `/content/`, se borra si cierras o reinicias el entorno.

## 9. Reto: aplica el mismo proceso a tu propio dataset

En la Clase 7 elegiste un dataset propio y llenaste una **ficha de auditoría de calidad** con los problemas que encontraste y las reglas de limpieza que propusiste. Hoy le toca a esas reglas convertirse en código.

Retoma ese dataset (o el notebook de la Clase 7 donde lo subiste) y aplica lo de hoy:

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# TODO: lee tu archivo
mis_datos = None

**Paso 1:** aplica al menos dos de las reglas de limpieza que propusiste en tu ficha de la Clase 7 (nulos, duplicados, tipos o inconsistencias).

In [ ]:
# TODO: aplica tus reglas de limpieza


**Paso 2 (el entregable de hoy):** crea **al menos dos variables analíticas nuevas** que permitan segmentar tus clientes, productos o servicios, por ejemplo, una categorización con `pd.cut()` (como `grupo_edad`), una columna calculada a partir de una fecha (como `antiguedad_dias`), o una columna enriquecida con `.map()` a partir de conocimiento externo (como `precio_plan_cop`).

In [ ]:
# TODO: crea al menos 2 variables analíticas nuevas para segmentar


**Paso 3:** guarda tu resultado en un nuevo archivo (nunca sobrescribas el original), con `to_csv()` o `to_excel()`.

In [ ]:
# TODO: guarda tu dataset transformado con otro nombre


## Cierre

Hoy cerraste el ciclo que empezó en la Clase 7: de un dataset con problemas diagnosticados, a un dataset corregido y enriquecido, siguiendo un pipeline reproducible (mismo orden de pasos, mismos parámetros fijos, archivo crudo intacto). Guarda una copia de este notebook y de tu `membresias_gimnasio_limpio.csv` en tu Google Drive.

En la **Clase 9: Agrupaciones y análisis con groupby** vas a usar exactamente las columnas que creaste hoy (`grupo_edad`, `ciudad`, `plan`, `antiguedad_categoria`) para responder preguntas del tipo "¿cuánto gasta en promedio cada grupo de edad?" o "¿qué ciudad tiene más socios activos?": la categorización de hoy es la materia prima del `groupby` de la próxima clase.